In [ ]:
import os
import cv2
import numpy as np

def remove_white_border(image_path, save_path=None, display=False, white_thresh=251):
    """
    Reads an image, finds the bounding box of non-white pixels, then crops out
    the white border area so the interior (plate + crack) remains.

    Args:
        image_path  (str): Path to the input image.
        save_path   (str): If provided, saves the cropped image to this path.
        display     (bool): Whether to display the cropped image in a pop-up window.
        white_thresh(int): Pixel intensity threshold to consider 'white'
                           (larger => more pixels treated as white).
    Returns:
        cropped_img (numpy array): The resulting cropped image.
    """
    # Reading the original image
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Could not read image from: {image_path}")

    # Convert to grayscale for easy intensity checks
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Create a mask of all "non-white" pixels
    mask_nonwhite = gray < white_thresh

    # Find the row/column indices where we actually have non-white pixels
    rows_with_nonwhite = np.where(np.any(mask_nonwhite, axis=1))[0]
    cols_with_nonwhite = np.where(np.any(mask_nonwhite, axis=0))[0]

    # Edge case: if the entire image is "white," just return original
    if len(rows_with_nonwhite) == 0 or len(cols_with_nonwhite) == 0:
        print(f"Warning: No non-white pixels found for {image_path}; returning original image.")
        return img

    # Compute the bounding box indices
    top, bottom = rows_with_nonwhite[0], rows_with_nonwhite[-1]
    left, right = cols_with_nonwhite[0], cols_with_nonwhite[-1]

    # Crop the image to that bounding box
    cropped_img = img[top:bottom+1, left:right+1]

    # Save the cropped image
    if save_path:
        cv2.imwrite(save_path, cropped_img)

    if display:
        cv2.imshow("Cropped Image", cropped_img)
        cv2.waitKey(0)
        cv2.destroyAllWindows()

    return cropped_img


def batch_preprocess_images(source_folder, destination_folder, white_thresh=251, display=False):
    os.makedirs(destination_folder, exist_ok=True)

    for filename in sorted(os.listdir(source_folder)):
        lower_fn = filename.lower()
        if not (lower_fn.endswith('.png') or lower_fn.endswith('.jpg') or lower_fn.endswith('.jpeg')):
            continue

        full_path = os.path.join(source_folder, filename)
        processed_filename = "Processed_" + filename
        processed_path = os.path.join(destination_folder, processed_filename)

        cropped = remove_white_border(
            image_path=full_path,
            save_path=processed_path,
            display=display,
            white_thresh=white_thresh
        )
        print(f"Saved processed image to: {processed_path}")


if __name__ == "__main__":
    input_folder = "Data/input"
    output_folder = "Data/output"

    processed_input_folder = "Processed Data/Processed Input"
    processed_output_folder = "Processed Data/Processed Output"

    # Process the input folder
    batch_preprocess_images(
        source_folder=input_folder,
        destination_folder=processed_input_folder,
        white_thresh=250,
        display=False
    )

    # Process the output folder
    batch_preprocess_images(
        source_folder=output_folder,
        destination_folder=processed_output_folder,
        white_thresh=250,
        display=False
    )

    print("All images processed. Check 'Processed Input' and 'Processed Output' folders.")

In [ ]:
# =========================
# VAE-based Crack Propagation Prediction with Physics-Informed Losses
# =========================

# 1. Imports
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
from torchvision.transforms.functional import gaussian_blur
import random
from skimage.metrics import structural_similarity as compare_ssim
from skimage.metrics import peak_signal_noise_ratio as compare_psnr
import lpips
from skimage.morphology import skeletonize
from skimage.measure import label, regionprops

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
lpips_model = lpips.LPIPS(net='alex').to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

# 2. SSIM Loss
def ssim_loss(pred, target, window_size=11, C1=0.01**2, C2=0.03**2):
    mu_x = gaussian_blur(pred, kernel_size=window_size)
    mu_y = gaussian_blur(target, kernel_size=window_size)

    mu_x_sq = mu_x ** 2
    mu_y_sq = mu_y ** 2
    mu_xy = mu_x * mu_y

    sigma_x_sq = gaussian_blur(pred ** 2, kernel_size=window_size) - mu_x_sq
    sigma_y_sq = gaussian_blur(target ** 2, kernel_size=window_size) - mu_y_sq
    sigma_xy = gaussian_blur(pred * target, kernel_size=window_size) - mu_xy

    ssim_n = (2 * mu_xy + C1) * (2 * sigma_xy + C2)
    ssim_d = (mu_x_sq + mu_y_sq + C1) * (sigma_x_sq + sigma_y_sq + C2)
    ssim_map = ssim_n / (ssim_d + 1e-7)

    return 1 - ssim_map.mean()

# 3. Physics-Informed Loss
class PhysicsLoss(nn.Module):
    def __init__(self, beta=0.3, gamma=0.2):
        super().__init__()
        self.beta = beta  # Growth constraint weight
        self.gamma = gamma  # Direction constraint weight

    def forward(self, pred, target, input_crack):
        growth_loss = self._crack_growth_constraint(pred, input_crack)
        direction_loss = self._direction_constraint(pred, input_crack)
        return self.beta * growth_loss + self.gamma * direction_loss

    def _crack_growth_constraint(self, pred, input_crack):
        # Use mean across channels to get single channel representation
        pred_binary = (pred.mean(dim=1, keepdim=True) > 0.5).float()
        input_binary = (input_crack.mean(dim=1, keepdim=True) > 0.5).float()
        growth_violation = torch.relu(input_binary - pred_binary)  # Only penalize shrinkage
        return torch.mean(growth_violation)

    def _direction_constraint(self, pred, input_crack):
        pred_np = pred.mean(dim=1).squeeze().detach().cpu().numpy()
        input_np = input_crack.mean(dim=1).squeeze().detach().cpu().numpy()

        if pred_np.ndim == 3:
            pred_np = pred_np[0]
        if input_np.ndim == 3:
            input_np = input_np[0]

        pred_tips = self._find_crack_tips(pred_np > 0.5)
        input_tips = self._find_crack_tips(input_np > 0.5)

        if len(pred_tips) == 0 or len(input_tips) == 0:
            return torch.tensor(0.0, dtype=torch.float32, device=pred.device)

        avg_direction = np.mean(pred_tips, axis=0) - np.mean(input_tips, axis=0)
        avg_direction_norm = avg_direction / (np.linalg.norm(avg_direction) + 1e-7)
        ideal_direction = np.array([1.0, 0.0])  # Assuming horizontal propagation
        direction_similarity = np.dot(avg_direction_norm, ideal_direction)

        return torch.tensor(1.0 - direction_similarity, dtype=torch.float32, device=pred.device)

    def _find_crack_tips(self, binary_image):
        if binary_image.ndim == 3:
            binary_image = binary_image[0]

        skeleton = skeletonize(binary_image)
        labeled = label(skeleton)
        tips = []

        for region in regionprops(labeled):
            coords = region.coords
            for coord in coords:
                y, x = coord[0], coord[1]
                neighbors = 0
                for dy, dx in [(-1,0), (1,0), (0,-1), (0,1), (-1,-1), (-1,1), (1,-1), (1,1)]:
                    if 0 <= y+dy < skeleton.shape[0] and 0 <= x+dx < skeleton.shape[1]:
                        neighbors += skeleton[y+dy, x+dx]
                if neighbors == 1:  # Crack tip has only one neighbor
                    tips.append([x, y])

        return np.array(tips) if tips else np.zeros((0, 2))

# 4. Dataset
class CrackDataset(Dataset):
    def __init__(self, input_dir, output_dir, target_size=(256, 256)):
        self.input_dir = input_dir
        self.output_dir = output_dir
        self.input_files = sorted(os.listdir(input_dir))
        self.output_files = sorted(os.listdir(output_dir))
        self.target_size = target_size
        assert len(self.input_files) == len(self.output_files), "Mismatched number of input/output files!"

    def __len__(self):
        return len(self.input_files)

    def __getitem__(self, idx):
        input_path = os.path.join(self.input_dir, self.input_files[idx])
        output_path = os.path.join(self.output_dir, self.output_files[idx])
        input_img = cv2.imread(input_path, cv2.IMREAD_COLOR)
        output_img = cv2.imread(output_path, cv2.IMREAD_COLOR)
        if input_img is None or output_img is None:
            raise ValueError(f"Could not read images at index {idx}")
        input_img = cv2.cvtColor(input_img, cv2.COLOR_BGR2RGB)
        output_img = cv2.cvtColor(output_img, cv2.COLOR_BGR2RGB)
        input_img = cv2.resize(input_img, self.target_size)
        output_img = cv2.resize(output_img, self.target_size)
        input_img = input_img.astype(np.float32) / 255.0
        output_img = output_img.astype(np.float32) / 255.0
        input_img = np.transpose(input_img, (2, 0, 1))
        output_img = np.transpose(output_img, (2, 0, 1))
        return {
            'input': torch.tensor(input_img, dtype=torch.float32),
            'output': torch.tensor(output_img, dtype=torch.float32)
        }

# 5. VAE Model
class VAE(nn.Module):
    def __init__(self, img_channels=3, latent_dim=128):
        super(VAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(img_channels, 32, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 256, 4, stride=2, padding=1),
            nn.ReLU(),
        )
        self.fc_mu = nn.Linear(256 * 16 * 16, latent_dim)
        self.fc_logvar = nn.Linear(256 * 16 * 16, latent_dim)
        self.fc_dec = nn.Linear(latent_dim, 256 * 16 * 16)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, img_channels, 4, stride=2, padding=1),
            nn.Sigmoid()
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        batch_size = x.size(0)
        encoded = self.encoder(x).view(batch_size, -1)
        mu = self.fc_mu(encoded)
        logvar = self.fc_logvar(encoded)
        z = self.reparameterize(mu, logvar)
        decoded = self.fc_dec(z).view(batch_size, 256, 16, 16)
        out = self.decoder(decoded)
        return out, mu, logvar

# 6. Combined VAE + Physics Loss
def vae_loss_function(recon, target, mu, logvar, input_crack, alpha=0.7, beta=0.2, gamma=0.1):
    # Standard VAE components
    ssim = ssim_loss(recon, target)
    kl_div = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    kl_div /= recon.size(0) * np.prod(recon.shape[1:])
    # Physics constraints
    physics_criterion = PhysicsLoss(beta=beta, gamma=gamma)
    physics_loss = physics_criterion(recon, target, input_crack)

    total_loss = alpha * ssim + (1-alpha) * kl_div + physics_loss

    return total_loss, ssim.item(), kl_div.item(), physics_loss.item()

# 7. Setup
input_dir = '/content/drive/MyDrive/Data/input'
output_dir = '/content/drive/MyDrive/Data/output'
dataset = CrackDataset(input_dir, output_dir)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(seed))
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

# 8. Training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = VAE().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5, verbose=True)

train_losses, val_losses = [], []
train_ssim_list, train_kl_list, train_physics_list = [], [], []

num_epochs = 500
best_val_loss = float('inf')

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        inputs = batch['input'].to(device)
        targets = batch['output'].to(device)

        optimizer.zero_grad()
        outputs, mu, logvar = model(inputs)
        loss, ssim_l, kl_l, physics_l = vae_loss_function(outputs, targets, mu, logvar, inputs)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * inputs.size(0)

    train_loss /= len(train_loader.dataset)
    train_ssim_list.append(ssim_l)
    train_kl_list.append(kl_l)
    train_physics_list.append(physics_l)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            inputs = batch['input'].to(device)
            targets = batch['output'].to(device)
            outputs, mu, logvar = model(inputs)
            loss, _, _, _ = vae_loss_function(outputs, targets, mu, logvar, inputs)
            val_loss += loss.item() * inputs.size(0)
    val_loss /= len(val_loader.dataset)
    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_crack_vae_physics.pth')
        print(f"Model saved at epoch {epoch+1}")

# 9. Plotting
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Training & Validation Loss')
plt.legend(); plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(train_ssim_list, label='SSIM Loss')
plt.plot(train_physics_list, label='Physics Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss Components'); plt.title('Loss Components Over Time')
plt.legend(); plt.grid(True)

plt.tight_layout()
plt.show()

# 10. Evaluation Metrics
def evaluate_model(model, loader):
    model.eval()
    mae_total, mse_total, ssim_total, psnr_total = 0, 0, 0, 0
    n = 0
    with torch.no_grad():
        for batch in loader:
            inputs = batch['input'].to(device)
            targets = batch['output'].to(device)
            outputs, _, _ = model(inputs)
            inputs_np = outputs.cpu().numpy()
            targets_np = targets.cpu().numpy()
            for i in range(inputs_np.shape[0]):
                pred_img = np.transpose(inputs_np[i], (1, 2, 0))
                true_img = np.transpose(targets_np[i], (1, 2, 0))
                mae_total += np.abs(pred_img - true_img).mean()
                mse_total += ((pred_img - true_img) ** 2).mean()
                ssim_total += compare_ssim(pred_img, true_img, channel_axis=-1, data_range=1.0)
                psnr_total += compare_psnr(true_img, pred_img, data_range=1.0)
            n += 1
    print(f"Final Validation Metrics:")
    print(f"  MAE  : {mae_total / n:.4f}")
    print(f"  MSE  : {mse_total / n:.4f}")
    print(f"  SSIM : {ssim_total / n:.4f}")
    print(f"  PSNR : {psnr_total / n:.2f} dB")

evaluate_model(model, val_loader)

def visualize_predictions(model, loader, num_samples=3):
    import cv2
    import numpy as np
    import matplotlib.pyplot as plt
    from skimage.metrics import structural_similarity as compare_ssim
    from skimage.metrics import peak_signal_noise_ratio as compare_psnr

    model.eval()
    device = next(model.parameters()).device
    shown = 0

    mae_list, mse_list, ssim_list, psnr_list = [], [], [], []
    dice_list, iou_list, lpips_list = [], [], []

    def enhance_cracks(img):
        img = (img * 255).astype(np.uint8)
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        no_grid = cv2.medianBlur(gray, 5)
        p1, p99 = np.percentile(no_grid, (1, 99))
        stretched = np.clip((no_grid - p1) * 255.0 / (p99 - p1 + 1e-5), 0, 255).astype(np.uint8)
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
        enhanced = cv2.dilate(stretched, kernel, iterations=1)
        return enhanced

    with torch.no_grad():
        for batch in loader:
            inputs = batch['input'].to(device)
            targets = batch['output'].to(device)
            outputs = model(inputs)[0]
            for i in range(inputs.size(0)):
                if shown >= num_samples:
                    break
                inp = inputs[i].cpu().permute(1, 2, 0).numpy()
                true = targets[i].cpu().permute(1, 2, 0).numpy()
                pred = outputs[i].cpu().permute(1, 2, 0).numpy()

                mae_list.append(np.abs(pred - true).mean())
                mse_list.append(((pred - true) ** 2).mean())
                ssim_list.append(compare_ssim(pred, true, channel_axis=-1, data_range=1.0))
                psnr_list.append(compare_psnr(true, pred, data_range=1.0))
                pred_bin = (pred.mean(axis=2) > 0.5).astype(np.uint8)
                true_bin = (true.mean(axis=2) > 0.5).astype(np.uint8)
                intersection = np.logical_and(pred_bin, true_bin).sum()
                union = np.logical_or(pred_bin, true_bin).sum()
                dice = (2. * intersection) / (pred_bin.sum() + true_bin.sum() + 1e-6)
                iou = intersection / (union + 1e-6)
                dice_list.append(dice)
                iou_list.append(iou)

                # LPIPS
                pred_t = torch.tensor(pred).permute(2, 0, 1).unsqueeze(0).to(device)
                true_t = torch.tensor(true).permute(2, 0, 1).unsqueeze(0).to(device)
                lpips_score = lpips_model(pred_t, true_t).item()
                lpips_list.append(lpips_score)

                inp_vis = enhance_cracks(inp)
                true_vis = enhance_cracks(true)
                pred_vis = enhance_cracks(pred)

                fig, axs = plt.subplots(1, 3, figsize=(15, 5))
                axs[0].imshow(inp); axs[0].set_title("Input"); axs[0].axis("off")
                axs[1].imshow(true_vis, cmap='gray'); axs[1].set_title("Ground Truth"); axs[1].axis("off")
                axs[2].imshow(pred_vis, cmap='gray'); axs[2].set_title("Prediction"); axs[2].axis("off")
                plt.tight_layout()
                plt.show()

                shown += 1
            if shown >= num_samples:
                break

    # Metrics summary
    print("\nValidation Sample Metrics Summary:")
    for i in range(shown):
        print(f"Sample {i+1}: MAE={mae_list[i]:.4f}, MSE={mse_list[i]:.4f}, SSIM={ssim_list[i]:.4f}, PSNR={psnr_list[i]:.2f} dB, Dice={dice_list[i]:.4f}, IoU={iou_list[i]:.4f}, LPIPS={lpips_list[i]:.4f}")
    print("-" * 50)
    print(f"Averages over {shown} samples:")
    print(f"  MAE   : {np.mean(mae_list):.4f}")
    print(f"  MSE   : {np.mean(mse_list):.4f}")
    print(f"  SSIM  : {np.mean(ssim_list):.4f}")
    print(f"  PSNR  : {np.mean(psnr_list):.2f} dB")
    print(f"  Dice  : {np.mean(dice_list):.4f}")
    print(f"  IoU   : {np.mean(iou_list):.4f}")
    print(f"  LPIPS : {np.mean(lpips_list):.4f}")

visualize_predictions(model, val_loader, num_samples=8)

In [ ]:
def visualize_predictions(model, loader, num_samples=3):
    import cv2
    import numpy as np
    import matplotlib.pyplot as plt
    from skimage.metrics import structural_similarity as compare_ssim
    from skimage.metrics import peak_signal_noise_ratio as compare_psnr

    model.eval()
    device = next(model.parameters()).device
    shown = 0

    mae_list, mse_list, ssim_list, psnr_list = [], [], [], []

    def enhance_cracks(img):
        img = (img * 255).astype(np.uint8)
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        no_grid = cv2.medianBlur(gray, 5)
        p1, p99 = np.percentile(no_grid, (1, 99))
        stretched = np.clip((no_grid - p1) * 255.0 / (p99 - p1 + 1e-5), 0, 255).astype(np.uint8)
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
        enhanced = cv2.dilate(stretched, kernel, iterations=1)
        return enhanced

    with torch.no_grad():
        for batch in loader:
            inputs = batch['input'].to(device)
            targets = batch['output'].to(device)
            outputs = model(inputs)[0]
            for i in range(inputs.size(0)):
                if shown >= num_samples:
                    break
                inp = inputs[i].cpu().permute(1, 2, 0).numpy()
                true = targets[i].cpu().permute(1, 2, 0).numpy()
                pred = outputs[i].cpu().permute(1, 2, 0).numpy()

                # Collect metrics
                mae_list.append(np.abs(pred - true).mean())
                mse_list.append(((pred - true) ** 2).mean())
                ssim_list.append(compare_ssim(pred, true, channel_axis=-1, data_range=1.0))
                psnr_list.append(compare_psnr(true, pred, data_range=1.0))

                # Show images
                inp_vis = enhance_cracks(inp)
                true_vis = enhance_cracks(true)
                pred_vis = enhance_cracks(pred)

                fig, axs = plt.subplots(1, 3, figsize=(15, 5))
                axs[0].imshow(inp); axs[0].set_title("Input"); axs[0].axis("off")
                axs[1].imshow(true_vis, cmap='gray'); axs[1].set_title("Ground Truth"); axs[1].axis("off")
                axs[2].imshow(pred_vis, cmap='gray'); axs[2].set_title("Prediction"); axs[2].axis("off")
                plt.tight_layout()
                plt.show()

                shown += 1
            if shown >= num_samples:
                break

    print("\nValidation Sample Metrics Summary:")
    for i in range(shown):
        print(f"Sample {i+1}: MAE={mae_list[i]:.4f}, MSE={mse_list[i]:.4f}, SSIM={ssim_list[i]:.4f}, PSNR={psnr_list[i]:.2f} dB")
    print("-" * 50)
    print(f"Averages over {shown} samples:")
    print(f"  MAE  : {np.mean(mae_list):.4f}")
    print(f"  MSE  : {np.mean(mse_list):.4f}")
    print(f"  SSIM : {np.mean(ssim_list):.4f}")
    print(f"  PSNR : {np.mean(psnr_list):.2f} dB")

visualize_predictions(model, val_loader, num_samples=8)
